# WasteWise VN — YOLO26 training
Chạy từ thư mục gốc repository. Notebook này dành cho thử nghiệm local; Kaggle GPU dùng bundle `ml/kaggle/kernel.py`. Giữ test đóng cho đến khi chọn xong cấu hình bằng validation.


In [ ]:
from pathlib import Path
import sys, subprocess, json, torch
ROOT = Path.cwd()
assert (ROOT / 'ml/scripts/train.py').is_file(), 'Open Jupyter from repository root'
DATA = ROOT / 'datasets/swim3_clean/data.yaml'
DEVICE = '0' if torch.cuda.is_available() else 'cpu'
print('Repository:', ROOT, 'device:', DEVICE)


In [ ]:
if not DATA.is_file():
    subprocess.run([sys.executable, 'ml/scripts/prepare_clean_split.py'], check=True)
subprocess.run([sys.executable, 'ml/scripts/audit_dataset.py', '--root', 'datasets/swim3_clean', '--out', 'outputs/notebook_audit.json'], check=True)


## GPU training — chạy sau khi đã đọc audit
Đặt `MODEL='yolo26s.pt'` cho run chính hoặc `yolo26n.pt` cho đối chứng. Cell dưới **sẽ train** khi bạn chạy.


In [ ]:
MODEL = 'yolo26s.pt'
RUN = MODEL.removesuffix('.pt')
subprocess.run([sys.executable, 'ml/scripts/train.py', '--data', str(DATA), '--model', MODEL, '--device', DEVICE, '--batch', '8', '--epochs', '80', '--name', RUN], check=True)


In [ ]:
BEST = ROOT / 'runs/train' / RUN / 'weights/best.pt'
subprocess.run([sys.executable, 'ml/scripts/evaluate.py', '--model', str(BEST), '--data', str(DATA), '--split', 'val', '--device', DEVICE, '--out', str(ROOT / 'outputs' / f'{RUN}_val.json')], check=True)
